# Занятие 2, пара 1. Камера. От света к пикселю

Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai

**Как проходить тетрадь.** Основные примеры работают независимо от заданий «Ваш ход». Если пока пропускаете задание, пропустите и его ячейку `check(...)`; продолжайте со следующего обычного раздела. Упражнения могут использовать ответы предыдущих упражнений. Сохраните свои попытки в отдельной копии тетради.

Как камера превращает большой предмет в маленькое изображение? Почему по размеру фотографии нельзя сразу узнать расстояние? И что ещё хранит JPEG, кроме самой картинки? Сегодня разберём эти вопросы без настоящей камеры и без нейросети.

Начнём с одного числа, затем нарисуем его смысл. В конце у вас будут расчёт проекции, сохранённая копия фото без EXIF и объяснение ограничения этой операции. Все примеры запускаются отдельно от первой пары.


Весь основной код работает без OpenCV: для массивов используем NumPy, для показа Matplotlib, для EXIF — Pillow. Тетрадь можно выполнять сверху вниз.

### Подключим учебные файлы

Следующая ячейка — служебная настройка комплекта. Она находит папку `course_support` у текущей папки или выше и подключает данные и проверки. Её не нужно писать по памяти или менять для задания. `DATA_DIR` — путь к учебным данным, `WORKSPACE_DIR` — корень вашей рабочей папки. `check(номер, ответы)` сравнивает ответ с условиями упражнения, но не выставляет оценку.

Работайте в своей копии и запускайте ячейки сверху вниз. Переменные из другой тетради не нужны. Если поддержку не удалось найти, откройте тетрадь внутри полного комплекта, а не переносите один файл отдельно. Git и интернет для основной работы не нужны.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

PROJECT_DIR


### Что подключаем

`numpy` хранит изображение как массив чисел, короткое имя — `np`. `matplotlib.pyplot` показывает картинки, короткое имя — `plt`. Для чтения EXIF ниже подключим Pillow. OpenCV для проекции, срезов и рисования квадрата не нужен.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

`/` между путём и строкой здесь соединяет папку и имя, а не делит числа. Результаты будем сохранять отдельно от исходных данных. `mkdir` создаёт папку; `parents=True` разрешает создать промежуточные папки, `exist_ok=True` не ругается, если папка уже есть.


In [ ]:
OUTPUT_DIR = WORKSPACE_DIR / 'reports' / 'camera'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


В папке camera лежат подготовленные учебные изображения. Снимок синтетический, настоящих сведений о человеке в нём нет. `plt.imread` читает файл; для нашего цветного JPEG возвращается массив uint8 в порядке RGB — красный, зелёный, синий. Позже в OpenCV встретится другой порядок, BGR; здесь каналы переставлять не надо.

`plt.imshow` показывает массив, `plt.title` задаёт подпись, `plt.show` выводит картинку. Посмотрим на неё до вычислений.

In [ ]:
CAMERA_DIR = DATA_DIR / 'camera'
image = plt.imread(CAMERA_DIR / 'teaching_photo.jpg')
plt.imshow(image)
plt.title('Учебный кадр')
plt.show()

`shape` — форма массива: высота, ширина, каналы. `dtype` — тип одного значения. uint8 означает целые 0…255. Индекс `[строка, столбец]` выбирает пиксель, поэтому `[100,200]` —100-я строка и 200-й столбец при счёте с нуля, а не x=100,y=200.


In [ ]:
print(image.shape)
print(image.dtype)
print(image[100, 200])


Срез `[100:200, 100:250]` выбирает строки 100…199 и столбцы 100…249. Конечная граница не включается. Возьмём участок и увидим, как изменился размер. Каналы оставляем все.


In [ ]:
crop = image[100:200, 100:250]
print(crop.shape)
plt.imshow(crop)
plt.title('Фрагмент')
plt.show()

**Попробуйте сами — 4 минуты.** Найдите другой участок изображения, получите его срез и предскажите shape до запуска. Не меняйте image: сохраняйте фрагмент в новой переменной. Если перепутались x и y, проверьте сначала срез только строк.


## Почему далеко — маленькое

Возьмём предмет высотой H=1 метр на расстоянии Z=2 метра. В простой модели его высота в кадре:

$$h_{px}=f_{px}\frac{H}{Z}.$$

`f_px` — фокусный масштаб в пикселях, пусть 500. H и Z должны быть в одинаковых единицах. Предмет расположен поперёк направления взгляда и целиком на одной глубине; для наклонного предмета простая формула уже не описывает весь случай.

Сначала просто числа: умножение `*`, деление `/`. Какое значение ожидаете?


In [ ]:
object_height = 1
focal_px = 500
distance = 2
height_px = focal_px * object_height / distance
print(height_px)


Теперь захотели отодвинуть тот же предмет. Меняем только расстояние. Прежде чем запускать: высота уменьшится вдвое или вчетверо? Формула описывает высоту; площадь квадрата ведёт себя иначе.


In [ ]:
distance = 4
height_far = focal_px * object_height / distance
print(height_far)
print('Отношение высот:', height_px / height_far)


Хотим повторять расчёт без копирования формулы. Объявим функцию через def: она получает высоту, расстояние и фокус. `focal_px=500` — значение по умолчанию. `return` возвращает результат. При нулевом или отрицательном расстоянии наша модель неприменима, поэтому `raise ValueError` сообщит причину остановки.


In [ ]:
def projected_height(object_height, distance, focal_px=500):
    if distance <= 0:
        raise ValueError('Расстояние должно быть положительным')
    height = focal_px * object_height / distance
    return height

print(projected_height(1, 2))


## Число есть. Хотим увидеть квадрат

Создадим белый холст 640×480. `np.full(форма, значение, dtype=...)` заполняет массив одним значением. Три канала по 255 дают белый цвет. Сначала создадим и покажем пустой холст; рисование добавим следующим шагом.


In [ ]:
canvas = np.full((480, 640, 3), 255, dtype=np.uint8)
plt.imshow(canvas)
plt.title('Пока только холст')
plt.show()

Квадрат расположим в центре кадра: x=320,y=240. От центра до каждой стороны — половина размера. Координаты для рисования нужны целые: `round` округляет, `int` превращает число в целое. Расчётная высота при этом может остаться дробной; рисунок лишь приближение на сетке пикселей.


In [ ]:
size = projected_height(1, 2)
half_size = size / 2
left = int(round(320 - half_size))
top = int(round(240 - half_size))
right = int(round(320 + half_size))
bottom = int(round(240 + half_size))
print(left, top, right, bottom)


Мы уже выбирали прямоугольный срез массива. Теперь запишем в него цвет: `[top:bottom, left:right]` — сначала строки, затем столбцы; правая и нижняя границы не включаются. Тройка `(40, 100, 180)` задаёт RGB. NumPy присвоит этот цвет всем пикселям выбранного участка. Это **меняет холст**, поэтому исходную фотографию для таких опытов надо копировать.

Срез заменяет вызов рисования из OpenCV; смысл геометрии от библиотеки не зависит.

In [ ]:
canvas[top:bottom, left:right] = (40, 100, 180)
plt.imshow(canvas)
plt.title('Квадрат при расстоянии 2 м')
plt.show()

Теперь соберём знакомые шаги в `draw_object`. В каждом вызове создаётся новый холст, поэтому прежний квадрат не остаётся на картинке. Для подписи используем `str(distance)`: число превращается в строку, а `+` соединяет строки. Функция покажет картинку и вернёт **расчётный** размер, который мы сможем сравнивать без округления.

В примере ниже расстояние 6 м — тот же вариант, который пробовали на занятии. Угадайте размер перед запуском.


У слишком близкого предмета границы выйдут за холст. `max(0, left)` выбирает большее из двух чисел, не пуская начало среза левее нуля; `min(640, right)` выбирает меньшее, ограничивая конец шириной холста. Так же ограничим верх и низ. Это обрезка рисунка по кадру, а не изменение расчётной высоты.

In [ ]:
def draw_object(distance, focal_px=500):
    size = projected_height(1, distance, focal_px)
    canvas = np.full((480, 640, 3), 255, dtype=np.uint8)
    half = size / 2
    left = int(round(320 - half))
    top = int(round(240 - half))
    right = int(round(320 + half))
    bottom = int(round(240 + half))
    # Ограничиваем срез краями холста: отрицательный индекс иначе считал бы с конца.
    left = max(0, left)
    top = max(0, top)
    right = min(640, right)
    bottom = min(480, bottom)
    canvas[top:bottom, left:right] = (40, 100, 180)
    title = 'Расстояние: ' + str(distance) + ' м'
    plt.imshow(canvas)
    plt.title(title)
    plt.show()
    return size

size_demo = draw_object(6)
print(size_demo)

### ✏️ Ваш ход — отодвинем предмет

Рассчитайте `near_size` для 2 м и `far_size` для 4 м через draw_object. Предмет 1 м, f=500px. Напечатайте отношение высот. Затем в отдельной ячейке при 4 м попробуйте f=1000. Что пришлось изменить, чтобы размер снова стал прежним?


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('02.1.1', near_size, far_size)


**Мой вывод:** замените эту строку своим объяснением результата. Назовите конкретное число, поле или участок изображения, на который опираетесь.


## Сведения рядом с пикселями

JPEG может содержать EXIF — метаданные о камере и съёмке. В нашем файле только вымышленные Make/Model/Software. Отсутствие EXIF тоже нормально.

Подключим Pillow: `Image.open` открывает файл, `getexif()` возвращает набор тегов. `with` удерживает файл открытым внутри блока с отступом и затем закрывает его. Сначала выведем номера тегов без украшений.


In [ ]:
from PIL import Image, ExifTags
photo_path = CAMERA_DIR / 'teaching_photo.jpg'
with Image.open(photo_path) as photo:
    exif = photo.getexif()
print(dict(exif))


Хотим имена вместо числовых кодов. `ExifTags.TAGS` — словарь известных имён. Метод `.get(ключ, запасное_значение)` вернёт имя либо исходный номер. `.items()` даёт пары «ключ, значение»; цикл распаковывает каждую пару в два имени. Сначала просто напечатаем каждую пару, без сложного словарного выражения.


In [ ]:
for tag_id, value in exif.items():
    tag_name = ExifTags.TAGS.get(tag_id, tag_id)
    print(tag_name, ':', value)


### Оставим только пиксели

`convert('RGB')` приводит изображение Pillow к трём каналам RGB. `np.array` получает массив значений. В этом массиве **нет EXIF**. Обратите внимание: у Pillow RGB, у OpenCV обычно BGR. Сейчас массив снова передадим Pillow, поэтому переставлять каналы не надо.


In [ ]:
with Image.open(photo_path) as photo:
    rgb_photo = photo.convert('RGB')
    pixels = np.array(rgb_photo)
print(pixels.shape)


`Image.fromarray` создаёт новое изображение из пикселей. Это новый объект без метаданных исходника. `.save(путь)` сохраняет его. Выбираем PNG, чтобы не добавлять новое JPEG-сжатие. Исходный JPEG не меняем.


In [ ]:
clean_image = Image.fromarray(pixels)
clean_path = OUTPUT_DIR / 'clean_photo.png'
clean_image.save(clean_path)
print(clean_path)


Файл записали — теперь надо открыть **именно записанный файл** и проверить. У Pillow свойство `.size` возвращает (ширину, высоту), обратный порядок относительно shape. `len` у набора EXIF посчитает теги. Всё для самостоятельной проверки уже разобрано.


### ✏️ Ваш ход — проверим сохранённую копию

Откройте clean_path через Image.open внутри with. Получите EXIF, сохраните число тегов как `clean_exif_count`, размер как `clean_size`. Покажите значения и откройте PNG обычным просмотрщиком. Объясните, почему отсутствие EXIF не скрывает лицо или номер, нарисованные на самой фотографии.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('02.1.2', clean_exif_count, clean_size)


**Мой вывод:** замените эту строку своим объяснением результата. Назовите конкретное число, поле или участок изображения, на который опираетесь.


## Если закончили раньше — один опыт на 10–15 минут

Подберите расстояние, при котором предмет 1 м при f=500px займёт 100px. Сначала оцените, потом проверьте projected_height. Теперь расстояние 4 м фиксировано — какой нужен f? Запишите два набора условий, которые дают один размер. Можно ли по одному размеру в кадре узнать расстояние, ничего больше не зная?

**Ещё один вопрос:** если размеры по высоте и ширине уменьшились в 2 раза, во сколько раз уменьшилась площадь квадрата? Проверьте умножением сторон, а не новым пакетом.

Проект: что хотите классифицировать, какие классы, откуда примеры? Пока достаточно записать идею и один следующий шаг. Публикация на GitHub добровольна.


## Итог

Сохраните тетрадь и `reports/camera/clean_photo.png`, сделайте отдельную копию. Сегодня разобрали изменение масштаба и различие пикселей и метаданных. Во второй паре выясним, почему прямые линии изгибаются и как оценить параметры камеры.

Для досуга — [«Фотоувеличение»](https://www.criterion.com/films/28694-blow-up), художественный фильм о наблюдении и интерпретации. Для дальнейшего чтения — [книга Сзелиски](https://szeliski.org/Book/), раздел о формировании изображения. Смотреть фильм и читать главу к следующей паре необязательно.


In [ ]:
summary('02.1')
